In [ ]:
#| default_exp models.cenn

In [ ]:
#| hide
%load_ext autoreload
%autoreload 2

In [ ]:
#| hide
import logging
import warnings
from fastcore.test import test_eq
from nbdev.showdoc import show_doc

In [ ]:
#| export
from typing import Optional
import torch.nn as nn

from neuralforecast.losses.pytorch import MAE
from neuralforecast.common._base_model import BaseModel

try:
    from cenn_forecasting import CeNNModel1D
except ImportError:
    try:
        from neuralforecast.cenn.model import CeNNModel1D
    except ImportError:
        CeNNModel1D = None

In [ ]:
#| export
class CeNN(BaseModel):
    """CeNN

    Cellular Neural Network (CeNN) wrapper compatible with NeuralForecast's BaseModel.

    Parameters:
    - h: int, forecast horizon.
    - input_size: int, input window length.
    - n_series: int, number of time-series (multivariate).
    - hidden_dim: int=64, latent dimension for internal CeNN representation.
    - N: int=2, number of CeNN blocks in the stack.
    - K: int=4, number of recurrent CeNN steps per layer.
    - num_layers: int=3, number of CeNN layers per block.
    - dropout: float=0.1, dropout rate.
    - neighborhood: int=3, convolution kernel size (must be odd).
    - alpha_init: float=0.9, initial leak parameter (0,1).
    - enforce_bistability: bool=False, clamp center tap in A kernel for stability (depthwise case). Disabled by default for regression tasks.
    - cross_channel: bool=False, allow full cross-feature CeNN coupling across series.
    - channel_groups: int=1, number of channels per group for grouped cross-channel mixing. 1=depthwise (default), >1=grouped conv.
    - adaptive_tau: bool=False, use input-adaptive integration gate (C1).
    - dilation_schedule: str='none', dilation schedule for multi-scale templates (C2). Options: 'none', 'exponential'.
    - var_mix: bool=False, enable dense cross-variable mixing (VarMix). Learns a V x V coupling matrix before embedding. Only effective when n_series > 1. Recommended for true multivariate datasets (ETT, Weather).
    - pointwise_mix: bool=False, enable pointwise Conv1d(C, C, 1) channel mixing after each CeNN layer's K-loop. Adds cross-channel interaction (MobileNet pattern).
    - patch_len: int=None, patch length for input patching. None=no patching (default).
    - stride: int=None, stride between patches. None=non-overlapping (patch_len).
    - head_type: str='linear', prediction head type. 'linear' or 'mlp'.
    - loss: PyTorch loss, defaults to MAE().
    - valid_loss: PyTorch loss used for validation, defaults to loss.
    - plus all standard BaseModel trainer and data args.
    """

    EXOGENOUS_FUTR = False
    EXOGENOUS_HIST = False
    EXOGENOUS_STAT = False
    MULTIVARIATE = True
    RECURRENT = False

    def __init__(
        self,
        h: int,
        input_size: int,
        n_series: int,
        stat_exog_list=None,
        hist_exog_list=None,
        futr_exog_list=None,
        exclude_insample_y: bool = False,
        # CeNN-specific
        hidden_dim: int = 64,
        N: int = 2,
        K: int = 10,
        num_layers: int = 3,
        dropout: float = 0.1,
        neighborhood: int = 3,
        alpha_init: float = 0.9,
        enforce_bistability: bool = False,
        cross_channel: bool = False,
        channel_groups: int = 1,
        adaptive_tau: bool = False,
        dilation_schedule: str = "none",
        var_mix: bool = False,
        pointwise_mix: bool = False,
        patch_len: Optional[int] = None,
        stride: Optional[int] = None,
        head_type: str = "linear",
        # Training
        loss: nn.Module = MAE(),
        valid_loss: Optional[nn.Module] = None,
        max_steps: int = 1000,
        learning_rate: float = 1e-3,
        num_lr_decays: int = -1,
        early_stop_patience_steps: int = -1,
        val_check_steps: int = 100,
        batch_size: int = 32,
        valid_batch_size: Optional[int] = None,
        windows_batch_size: int = 1024,
        inference_windows_batch_size: int = 1024,
        start_padding_enabled: bool = False,
        training_data_availability_threshold: float = 0.0,
        step_size: int = 1,
        scaler_type: str = "identity",
        random_seed: int = 1,
        drop_last_loader: bool = False,
        alias: Optional[str] = None,
        optimizer=None,
        optimizer_kwargs=None,
        lr_scheduler=None,
        lr_scheduler_kwargs=None,
        dataloader_kwargs=None,
        **trainer_kwargs,
    ):
        super().__init__(
            h=h,
            input_size=input_size,
            n_series=n_series,
            stat_exog_list=stat_exog_list,
            hist_exog_list=hist_exog_list,
            futr_exog_list=futr_exog_list,
            exclude_insample_y=exclude_insample_y,
            loss=loss,
            valid_loss=valid_loss if valid_loss is not None else loss,
            max_steps=max_steps,
            learning_rate=learning_rate,
            num_lr_decays=num_lr_decays,
            early_stop_patience_steps=early_stop_patience_steps,
            val_check_steps=val_check_steps,
            batch_size=batch_size,
            valid_batch_size=valid_batch_size,
            windows_batch_size=windows_batch_size,
            inference_windows_batch_size=inference_windows_batch_size,
            start_padding_enabled=start_padding_enabled,
            training_data_availability_threshold=training_data_availability_threshold,
            step_size=step_size,
            scaler_type=scaler_type,
            random_seed=random_seed,
            drop_last_loader=drop_last_loader,
            alias=alias,
            optimizer=optimizer,
            optimizer_kwargs=optimizer_kwargs,
            lr_scheduler=lr_scheduler,
            lr_scheduler_kwargs=lr_scheduler_kwargs,
            dataloader_kwargs=dataloader_kwargs,
            **trainer_kwargs,
        )

        self.c_out = self.loss.outputsize_multiplier

        # Inner CeNN model (multivariate)
        self.model = CeNNModel1D(
            n_features=n_series,
            seq_length=self.input_size,
            pred_length=self.h,
            hidden_dim=hidden_dim,
            N=N,
            K=K,
            dropout=dropout,
            num_layers=num_layers,
            neighborhood=neighborhood,
            alpha_init=alpha_init,
            enforce_bistability=enforce_bistability,
            cross_channel=cross_channel,
            channel_groups=channel_groups,
            adaptive_tau=adaptive_tau,
            dilation_schedule=dilation_schedule,
            var_mix=var_mix,
            pointwise_mix=pointwise_mix,
            patch_len=patch_len,
            stride=stride,
            head_type=head_type,
            output_multiplier=self.c_out,
        )

    def forward(self, windows_batch):
        # Extract historic values (insample_y) with shape [B, L, n_series]
        x = windows_batch["insample_y"]  # [B, L, n_series]
        out = self.model(x)  # [B, h, n_series * c_out]
        return out

In [ ]:
show_doc(CeNN)

In [ ]:
show_doc(CeNN.fit, name='CeNN.fit')


In [ ]:
show_doc(CeNN.predict, name='CeNN.predict')

## Usage Example
This example shows how to train and forecast with the multivariate CeNN model using the AirPassengers panel dataset. It’s disabled from execution in docs.

In [ ]:
#| eval: false
import pandas as pd
import matplotlib.pyplot as plt

from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MAE
from neuralforecast.utils import AirPassengersPanel, AirPassengersStatic
from neuralforecast.models import CeNN

# Split train/test
Y_train_df = AirPassengersPanel[AirPassengersPanel.ds < AirPassengersPanel['ds'].values[-12]]  # 132 train
Y_test_df = AirPassengersPanel[AirPassengersPanel.ds >= AirPassengersPanel['ds'].values[-12]].reset_index(drop=True)  # 12 test

# Instantiate CeNN (multivariate)
CeNN = CeNN(
    h=12,
    input_size=24,
    n_series=2,
    loss=MAE(),
    scaler_type='robust',
    learning_rate=1e-3,
    max_steps=1000,
    val_check_steps=10,
    early_stop_patience_steps=2,
    hidden_dim=128,
    N=2,
    K=10,
    num_layers=3,
    dropout=0.3,
    neighborhood=3,
    alpha_init=0.9,
    enforce_bistability=True,
    cross_channel=False,
    accelerator='gpu',
    enable_progress_bar=True
)

# Fit and predict
fcst = NeuralForecast(models=[CeNN], freq='ME')
fcst.fit(df=Y_train_df, static_df=AirPassengersStatic, val_size=12)
forecasts = fcst.predict(futr_df=Y_test_df)

# Plot predictions
Y_hat_df = forecasts.reset_index(drop=False).drop(columns=['unique_id','ds'])
plot_df = pd.concat([Y_test_df, Y_hat_df], axis=1)
plot_df = pd.concat([Y_train_df, plot_df])

plot_df = plot_df[plot_df.unique_id=='Airline1'].drop('unique_id', axis=1)
plt.figure(figsize=(12,6))
plt.plot(plot_df['ds'], plot_df['y'], c='black', label='True', linewidth=2)
plt.plot(plot_df['ds'], plot_df['CeNN'], c='red', label='CeNN', linewidth=1.5)
plt.grid(True, alpha=0.3)
plt.legend()
plt.title('Time Series Forecasting: CeNN')
plt.xlabel('Date')
plt.ylabel('Values')
plt.show()